In [7]:
import boto3
import torch
import numpy as np

from PIL import Image
from io import BytesIO
from torchvision import transforms

session = boto3.Session(profile_name="ATLAS")
s3 = session.client("s3", region_name="us-east-2")

SOURCE_BUCKET = "atlas-acm-research-f26"
SOURCE_PREFIX = "satellite/raw/"

print("AWS connected successfully")

AWS connected successfully


In [8]:
#retrieve image from S3
image_keys = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket=SOURCE_BUCKET,
    Prefix=SOURCE_PREFIX
):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        if key.lower().endswith(".png"):
            image_keys.append(key)

image_keys.sort()

print("Total images found:", len(image_keys))
print("First 5 images:", image_keys[:5])

Total images found: 250
First 5 images: ['satellite/raw/0052227492.png', 'satellite/raw/0169170979.png', 'satellite/raw/0209073657.png', 'satellite/raw/0239032142.png', 'satellite/raw/0283051477.png']


In [9]:
#image to tensor preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [10]:
#download and convert images 
key = image_keys[0]

response = s3.get_object(
    Bucket=SOURCE_BUCKET,
    Key=key
)

image_bytes = response["Body"].read()
image = Image.open(BytesIO(image_bytes)).convert("RGB")

tensor = transform(image)

image_id = key.rsplit("/", 1)[-1].removesuffix(".png")

print("Dataset ID:", image_id)
print("Original image size:", image.size)
print("Tensor shape:", tensor.shape)
print("Tensor dtype:", tensor.dtype)
print("Tensor range:", tensor.min().item(), tensor.max().item())

Dataset ID: 0052227492
Original image size: (1000, 842)
Tensor shape: torch.Size([3, 224, 224])
Tensor dtype: torch.float32
Tensor range: 0.08627451211214066 0.9215686321258545


In [11]:
DEST_BUCKET = "atlas-acm-research-f26"
DEST_PREFIX = "satellite/processed/"

In [ ]:
import io

uploaded = 0
failed = []

for i, key in enumerate(image_keys, start=1):
    image_id = key.rsplit("/", 1)[-1].removesuffix(".png")
    tensor_key = f"{DEST_PREFIX}{image_id}.pt"

    try:
        # download PNG from source S3 bucket
        response = s3.get_object(
            Bucket=SOURCE_BUCKET,
            Key=key
        )

        image = Image.open(
            BytesIO(response["Body"].read())
        ).convert("RGB")

        # convert image into PyTorch tensor
        tensor = transform(image)

        # serialize tensor in memory
        buffer = io.BytesIO()
        torch.save({
            "id": image_id,
            "image": tensor,
            "source_key": key
        }, buffer)

        buffer.seek(0)

        # upload .pt file to destination S3 bucket
        s3.upload_fileobj(
            buffer,
            DEST_BUCKET,
            tensor_key
        )

        uploaded += 1
        print(f"[{i}/{len(image_keys)}] Uploaded: {tensor_key}")

    except Exception as e:
        failed.append((image_id, str(e)))
        print(f"[{i}/{len(image_keys)}] ERROR {image_id}: {e}")

print(f"\nSuccessfully uploaded: {uploaded}")
print(f"Failed: {len(failed)}")

[1/250] Uploaded: satellite/processed/0052227492.pt
[2/250] Uploaded: satellite/processed/0169170979.pt
[3/250] Uploaded: satellite/processed/0209073657.pt
[4/250] Uploaded: satellite/processed/0239032142.pt
[5/250] Uploaded: satellite/processed/0283051477.pt
[6/250] Uploaded: satellite/processed/0283051485.pt
[7/250] Uploaded: satellite/processed/0299929061.pt
[8/250] Uploaded: satellite/processed/0337127832.pt
[9/250] Uploaded: satellite/processed/0435817145.pt
[10/250] Uploaded: satellite/processed/0610827836.pt
[11/250] Uploaded: satellite/processed/0673532692.pt
[12/250] Uploaded: satellite/processed/0696985482.pt
[13/250] Uploaded: satellite/processed/0751881301.pt
[14/250] Uploaded: satellite/processed/0762116506.pt
[15/250] Uploaded: satellite/processed/0781370600.pt
[16/250] Uploaded: satellite/processed/0830885774.pt
[17/250] Uploaded: satellite/processed/0878795698.pt
[18/250] Uploaded: satellite/processed/0944024849.pt
[19/250] Uploaded: satellite/processed/0977653854.pt
[2